In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    StratifiedKFold, StratifiedShuffleSplit, 
    RepeatedStratifiedKFold, LeaveOneOut,
    cross_validate
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

class AdvancedCrossValidation:
    def __init__(self, csv_path, random_state=42):
        self.csv_path = csv_path
        self.random_state = random_state
        self.results = {}
        self.df = None
        self.labeled_data = None
        self.unlabeled_data = None
        self.X_labeled = None
        self.y_labeled_encoded = None
        self.class_names = None
        self.feature_names = None

    def load_and_prepare_data(self):
        """Load CSV and separate labeled from unlabeled data"""
        print("🔄 Loading and preparing data...")
        
        # Load CSV
        self.df = pd.read_csv(self.csv_path)
        print(f"✅ Data loaded successfully! Total Shape: {self.df.shape}")
        
        # Check for 'layer' column
        if 'layer' not in self.df.columns:
            raise ValueError("❌ 'layer' column not found in the dataset!")
        
        # Separate labeled and unlabeled data
        labeled_mask = (
            self.df['layer'].notna() & 
            (self.df['layer'] != '') & 
            (self.df['layer'] != 'Unknown')
        )
        
        self.labeled_data = self.df[labeled_mask].copy()
        self.unlabeled_data = self.df[~labeled_mask].copy()
        
        print(f"📊 Labeled samples: {len(self.labeled_data)}")
        print(f"🔍 Unlabeled samples: {len(self.unlabeled_data)}")
        print(f"🎯 Target classes in labeled data: {sorted(self.labeled_data['layer'].unique())}")
        
        if len(self.labeled_data) == 0:
            raise ValueError("❌ No labeled data found for training!")
        
        # Identify feature columns (exclude geometry, label, layer)
        exclude_cols = ['geometry', 'label', 'layer']
        feature_cols = [col for col in self.df.columns if col not in exclude_cols]
        
        if len(feature_cols) == 0:
            raise ValueError("❌ No feature columns found after excluding geometry, label, and layer!")
        
        print(f"🔧 Found {len(feature_cols)} feature columns")
        
        # Prepare features and target for labeled data
        self.X_labeled = self.labeled_data[feature_cols].fillna(0)
        self.y_labeled = self.labeled_data['layer']
        
        # Prepare features for unlabeled data
        if len(self.unlabeled_data) > 0:
            self.X_unlabeled = self.unlabeled_data[feature_cols].fillna(0)
        
        self.feature_names = feature_cols
        
        # Encode labels if they are strings
        if self.y_labeled.dtype == 'object':
            self.label_encoder = LabelEncoder()
            self.y_labeled_encoded = self.label_encoder.fit_transform(self.y_labeled)
            self.class_names = list(self.label_encoder.classes_)
        else:
            self.y_labeled_encoded = self.y_labeled
            self.class_names = sorted(self.y_labeled.unique())
        
        print(f"✅ Data preparation completed!")
        print(f"   Classes: {self.class_names}")
        return self

    def stratified_k_fold(self, k=5, n_repeats=1):
        if n_repeats > 1:
            cv = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=self.random_state)
            cv_name = f"RepeatedStratifiedKFold_{k}x{n_repeats}"
        else:
            cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=self.random_state)
            cv_name = f"StratifiedKFold_{k}"
        return self._run_cv(cv, cv_name)
    
    def stratified_shuffle_split(self, n_splits=10, test_size=0.3):
        cv = StratifiedShuffleSplit(
            n_splits=n_splits, 
            test_size=test_size, 
            random_state=self.random_state
        )
        return self._run_cv(cv, f"StratifiedShuffleSplit_{n_splits}")
    
    def leave_one_out_cv(self):
        cv = LeaveOneOut()
        return self._run_cv(cv, "LeaveOneOut", verbose=False)
    
    def monte_carlo_cv(self, n_iterations=50, test_size=0.3):
        cv = StratifiedShuffleSplit(
            n_splits=n_iterations,
            test_size=test_size,
            random_state=self.random_state
        )
        return self._run_cv(cv, f"MonteCarlo_{n_iterations}")
    
    def _run_cv(self, cv, cv_name, verbose=True):
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=self.random_state,
            n_jobs=-1
        )
        
        cv_results = cross_validate(
            rf, self.X_labeled, self.y_labeled_encoded, cv=cv,
            scoring=['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'],
            return_train_score=True,
            n_jobs=-1
        )
        
        results = {
            'cv_strategy': cv_name,
            'n_splits': cv.get_n_splits(self.X_labeled, self.y_labeled_encoded),
            'test_accuracy_mean': cv_results['test_accuracy'].mean(),
            'test_accuracy_std': cv_results['test_accuracy'].std(),
            'test_f1_mean': cv_results['test_f1_macro'].mean(),
            'test_f1_std': cv_results['test_f1_macro'].std(),
            'test_precision_mean': cv_results['test_precision_macro'].mean(),
            'test_precision_std': cv_results['test_precision_macro'].std(),
            'test_recall_mean': cv_results['test_recall_macro'].mean(),
            'test_recall_std': cv_results['test_recall_macro'].std(),
            'train_accuracy_mean': cv_results['train_accuracy'].mean(),
            'train_accuracy_std': cv_results['train_accuracy'].std(),
            'overfitting_gap': cv_results['train_accuracy'].mean() - cv_results['test_accuracy'].mean(),
            'all_scores': cv_results
        }
        
        self.results[cv_name] = results
        
        if verbose:
            print(f"✅ {cv_name} Results:")
            print(f"   Test Accuracy: {results['test_accuracy_mean']:.4f} ± {results['test_accuracy_std']:.4f}")
            print(f"   Test F1-Score: {results['test_f1_mean']:.4f} ± {results['test_f1_std']:.4f}")
            print(f"   Overfitting Gap: {results['overfitting_gap']:.4f}")
            print()
        
        return results
    
    def compare_all_strategies(self):
        print("🚀 Comparing Multiple Cross-Validation Strategies...")
        print("="*60)
        
        strategies = [
            ('Stratified 5-Fold', lambda: self.stratified_k_fold(k=5)),
            ('Stratified 10-Fold', lambda: self.stratified_k_fold(k=10)),
            ('Repeated 5-Fold (3x)', lambda: self.stratified_k_fold(k=5, n_repeats=3)),
            ('Shuffle Split (20 splits)', lambda: self.stratified_shuffle_split(n_splits=20)),
            ('Monte Carlo (30 iter)', lambda: self.monte_carlo_cv(n_iterations=30)),
        ]
        
        for name, strategy in strategies:
            try:
                strategy()
            except Exception as e:
                print(f"❌ {name} failed: {str(e)}")
        
        comparison_data = []
        for cv_name, results in self.results.items():
            comparison_data.append({
                'Strategy': cv_name,
                'N_Splits': results['n_splits'],
                'Test_Accuracy_Mean': results['test_accuracy_mean'],
                'Test_Accuracy_Std': results['test_accuracy_std'],
                'Test_F1_Mean': results['test_f1_mean'],
                'Test_F1_Std': results['test_f1_std'],
                'Overfitting_Gap': results['overfitting_gap']
            })
        
        comparison_df = pd.DataFrame(comparison_data)
        comparison_df = comparison_df.sort_values('Test_Accuracy_Mean', ascending=False)
        
        print("📊 CROSS-VALIDATION COMPARISON:")
        print("="*80)
        print(comparison_df.round(4).to_string(index=False))
        print("="*80)
        
        best_strategy = comparison_df.iloc[0]
        print(f"🏆 BEST STRATEGY: {best_strategy['Strategy']}")
        print(f"   Accuracy: {best_strategy['Test_Accuracy_Mean']:.4f} ± {best_strategy['Test_Accuracy_Std']:.4f}")
        print(f"   F1-Score: {best_strategy['Test_F1_Mean']:.4f} ± {best_strategy['Test_F1_Std']:.4f}")
        print(f"   Overfitting: {best_strategy['Overfitting_Gap']:.4f}")
        
        return comparison_df


In [2]:
cv_analyzer = AdvancedCrossValidation(r"F:\_____My_Thesies____\_ImpFiles\4. LULC\1.Feature_Engineered\2.RawFeature_AllClass\final_dataset_all_ReduceClassesAsLatifiSaid.csv")
cv_analyzer.load_and_prepare_data()
results = cv_analyzer.compare_all_strategies()

🔄 Loading and preparing data...
✅ Data loaded successfully! Total Shape: (6318, 2782)
📊 Labeled samples: 431
🔍 Unlabeled samples: 5887
🎯 Target classes in labeled data: ['Alfalfa-polygon', 'Bare-polygon', 'Clover-polygon', 'Rice-polygon', 'Urban-polygon', 'WetlandVeg-polygon', 'Wheat-polygon']
🔧 Found 2780 feature columns
✅ Data preparation completed!
   Classes: ['Alfalfa-polygon', 'Bare-polygon', 'Clover-polygon', 'Rice-polygon', 'Urban-polygon', 'WetlandVeg-polygon', 'Wheat-polygon']
🚀 Comparing Multiple Cross-Validation Strategies...
✅ StratifiedKFold_5 Results:
   Test Accuracy: 0.9582 ± 0.0158
   Test F1-Score: 0.9264 ± 0.0344
   Overfitting Gap: 0.0348

✅ StratifiedKFold_10 Results:
   Test Accuracy: 0.9582 ± 0.0251
   Test F1-Score: 0.9205 ± 0.0443
   Overfitting Gap: 0.0351

✅ RepeatedStratifiedKFold_5x3 Results:
   Test Accuracy: 0.9598 ± 0.0134
   Test F1-Score: 0.9246 ± 0.0339
   Overfitting Gap: 0.0325

✅ StratifiedShuffleSplit_20 Results:
   Test Accuracy: 0.9573 ± 0.0145